In [ ]:
import os
import sys
os.chdir("/workspaces/dev")
paths = [
    "/workspaces/dev/modules/python-utils",
    "/workspaces/dev/modules/ai-utils",
    "/workspaces/dev/test/performance_test/libri",
]
for path in paths:
    sys.path.append(os.path.abspath(path))
print(f"Current Python version: {sys.version}")

In [ ]:
from pathlib import Path

In [ ]:
from sj_ai_utils.datasets.esic_v1.service import search_dirs
from sj_utils.file.yaml import load_yaml
from sj_utils.file.json import JsonSaver
from sj_utils.collection import SafetyDict

In [ ]:
from summarize import whisper, rt_whisper, whisper_streaming, chunk_test, hyperparameter_test, generate_statistical_dict, show_table, show_ribbon_plot, show_line_plot

In [ ]:
RANDOM_SEED = 42
MAX_COUNT = 1
REPEAT = 1
TEST_STEP = 1

In [ ]:
DESCRIPTION = """
TEST
"""

In [ ]:
TABLE_KEY = ["Model", "WER (%)", "Latency Mean (s)", "Latency Q1", "Latency Q2", "Latency Q3", "Latency Max"]

In [ ]:
SOURCE = "/workspaces/dev/datasets/LibriSpeechASRcorpus/dev-clean"
STORAGE = "/workspaces/dev/storage/libri/"
HYPERPARAMETERS_PATH = "/workspaces/dev/test/performance_test/esic/hyperparameters/20250731/step1_16b-96k/trial_wer3o0_6690_20250801_074742.yaml"
JSON_SAVE_PATH = "/workspaces/dev/test/performance_test/libri/summarize_result.json"

In [ ]:
source = Path(SOURCE)
storage = Path(STORAGE)
json_save_path = Path(JSON_SAVE_PATH)
hyperparameter_path = Path(HYPERPARAMETERS_PATH)
if not hyperparameter_path.exists():
    raise FileNotFoundError(f"Hyperparameter file not found: {hyperparameter_path}")
if not source.exists():
    raise FileNotFoundError(f"Source directory not found: {source}")

In [ ]:
json_saver = JsonSaver(DESCRIPTION)

In [ ]:
data_dirs = search_dirs(source)

In [ ]:
_, hyperparameter_original = load_yaml(hyperparameter_path)
hyperparameter = SafetyDict(hyperparameter_original)

In [ ]:
def results_to_table_data(result: list[dict]):
    data = [
        [
            r["wer_percent"],
            r["transcribe_time"]["mean"],
            r["transcribe_time"]["q1"],
            r["transcribe_time"]["q2"],
            r["transcribe_time"]["q3"],
            r["transcribe_time"]["max"],
        ]
        for r in result
    ]

    return generate_statistical_dict(data)

def result_to_table_data(result: dict):
    return [
        f"{result['wer_percent']:.2f}",
        f"{result['transcribe_time']['mean']:.2f}",
        f"{result['transcribe_time']['q1']:.2f}",
        f"{result['transcribe_time']['q2']:.2f}",
        f"{result['transcribe_time']['q3']:.2f}",
        f"{result['transcribe_time']['max']:.2f}",
    ]

In [ ]:
whisper_result = whisper(data_dirs, repeat=REPEAT, count=MAX_COUNT)
rt_whisper_result = rt_whisper(data_dirs, hyperparameter, 48000, 400, 0.1, repeat=REPEAT, count=MAX_COUNT, random_seed = RANDOM_SEED)
whisper_streaming_result = whisper_streaming(data_dirs, 48000, 400, 0.1, repeat=REPEAT, count=MAX_COUNT, random_seed = RANDOM_SEED)

In [ ]:
chunk_result = chunk_test(data_dirs, hyperparameter, 8000, 8000, 64000, count=MAX_COUNT)

In [ ]:
hyperparameter_result = hyperparameter_test(TEST_STEP, data_dirs, hyperparameter_original, 48000, count=MAX_COUNT)

In [ ]:
json_saver.save(
    {
        "whisper": whisper_result,
        "rt_whisper": rt_whisper_result,
        "whisper_streaming": whisper_streaming_result,
        "chunk": chunk_result,
        "hyperparameter": hyperparameter_result,
    }, json_save_path
)

In [ ]:
print("Whisper Result:", whisper_result)
print("RT Whisper Result:", rt_whisper_result)
print("Whisper Streaming Result:", whisper_streaming_result)
print("Chunk Result:", chunk_result)
print("Hyperparameter Result:", hyperparameter_result)

In [ ]:
show_table(
    [*TABLE_KEY],
    [
        ["whisper", *results_to_table_data(whisper_result)],
        ["rt whisper", *results_to_table_data(rt_whisper_result)],
        ["whisper streaming", *results_to_table_data(whisper_streaming_result)],
    ]
)

In [ ]:
x = list(chunk_result.keys())
rt_whisper_wer = [chunk_result[chunk_size]["rt_whisper"]["wer_percent"] for chunk_size in x]
rt_whisper_latency_mean = [chunk_result[chunk_size]["rt_whisper"]["transcribe_time"]["mean"] for chunk_size in x]
rt_whisper_latency_min = [chunk_result[chunk_size]["rt_whisper"]["transcribe_time"]["min"] for chunk_size in x]
rt_whisper_latency_q1 = [chunk_result[chunk_size]["rt_whisper"]["transcribe_time"]["q1"] for chunk_size in x]
rt_whisper_latency_q2 = [chunk_result[chunk_size]["rt_whisper"]["transcribe_time"]["q2"] for chunk_size in x]
rt_whisper_latency_q3 = [chunk_result[chunk_size]["rt_whisper"]["transcribe_time"]["q3"] for chunk_size in x]
rt_whisper_latency_max = [chunk_result[chunk_size]["rt_whisper"]["transcribe_time"]["max"] for chunk_size in x]

whisper_streaming_wer = [chunk_result[chunk_size]["whisper_streaming"]["wer_percent"] for chunk_size in x]
whisper_streaming_latency_mean = [chunk_result[chunk_size]["whisper_streaming"]["transcribe_time"]["mean"] for chunk_size in x]
whisper_streaming_latency_min = [chunk_result[chunk_size]["whisper_streaming"]["transcribe_time"]["min"] for chunk_size in x]
whisper_streaming_latency_q1 = [chunk_result[chunk_size]["whisper_streaming"]["transcribe_time"]["q1"] for chunk_size in x]
whisper_streaming_latency_q2 = [chunk_result[chunk_size]["whisper_streaming"]["transcribe_time"]["q2"] for chunk_size in x]
whisper_streaming_latency_q3 = [chunk_result[chunk_size]["whisper_streaming"]["transcribe_time"]["q3"] for chunk_size in x]
whisper_streaming_latency_max = [chunk_result[chunk_size]["whisper_streaming"]["transcribe_time"]["max"] for chunk_size in x]

In [ ]:
show_line_plot(
    x, rt_whisper_wer, whisper_streaming_wer,
    "RT Whisper vs Whisper Streaming WER",
    "Chunk Size (samples)", "WER (%)",
    "RT Whisper", "Whisper Streaming",
)

In [ ]:
show_ribbon_plot(
    x,
    rt_whisper_latency_mean, rt_whisper_latency_q1, rt_whisper_latency_q2, rt_whisper_latency_q3,
    whisper_streaming_latency_mean, whisper_streaming_latency_q1, whisper_streaming_latency_q2, whisper_streaming_latency_q3,
    "RT Whisper Latency Distribution by Chunk Size",
    "Chunk Size (samples)", "Latency (s)",
    "RT Whisper", "Whisper Streaming",
)

In [ ]:
rt_whisper_wer =rt_whisper_result[0]["wer_percent"]
rt_whisper_latency_mean = rt_whisper_result[0]["transcribe_time"]["mean"]
rt_whisper_latency_q1 = rt_whisper_result[0]["transcribe_time"]["q1"]
rt_whisper_latency_q2 = rt_whisper_result[0]["transcribe_time"]["q2"]
rt_whisper_latency_q3 = rt_whisper_result[0]["transcribe_time"]["q3"]

In [ ]:
data = hyperparameter_result["max_overlap_duration"]
x = list(data.keys())
wer = [data[od]["wer_percent"] for od in x]
latency_mean = [data[od]["transcribe_time"]["mean"] for od in x]
latency_q1 = [data[od]["transcribe_time"]["q1"] for od in x]
latency_q2 = [data[od]["transcribe_time"]["q2"] for od in x]
latency_q3 = [data[od]["transcribe_time"]["q3"] for od in x]

show_line_plot(
    x,
    wer, [rt_whisper_wer for _ in range(len(x))],
    "Max Overlap Duration WER Comparison",
    "Max Overlap Duration (samples)", "WER (%)",
    "Max Overlap Duration", "RT Whisper",
)
show_ribbon_plot(
    x,
    latency_mean, latency_q1, latency_q2, latency_q3,
    [rt_whisper_latency_mean for _ in range(len(x))], [rt_whisper_latency_q1 for _ in range(len(x))], [rt_whisper_latency_q2 for _ in range(len(x))], [rt_whisper_latency_q3 for _ in range(len(x))],
    "Max Overlap Duration Latency Distribution",
    "Max Overlap Duration (samples)", "Latency (s)",
    "Max Overlap Duration", "RT Whisper",
)

In [ ]:
data = hyperparameter_result["max_prompt_words"]
x = list(data.keys())
wer = [data[mpw]["wer_percent"] for mpw in x]
latency_mean = [data[mpw]["transcribe_time"]["mean"] for mpw in x]
latency_q1 = [data[mpw]["transcribe_time"]["q1"] for mpw in x]
latency_q2 = [data[mpw]["transcribe_time"]["q2"] for mpw in x]
latency_q3 = [data[mpw]["transcribe_time"]["q3"] for mpw in x]

show_line_plot(
    x,
    wer, [rt_whisper_wer for _ in range(len(x))],
    "Max Prompt Words WER Comparison",
    "Max Prompt Words", "WER (%)",
    "Max Prompt Words", "RT Whisper",
)

show_ribbon_plot(
    x,
    latency_mean, latency_q1, latency_q2, latency_q3,
    [rt_whisper_latency_mean for _ in range(len(x))], [rt_whisper_latency_q1 for _ in range(len(x))], [rt_whisper_latency_q2 for _ in range(len(x))], [rt_whisper_latency_q3 for _ in range(len(x))],
    "Max Prompt Words Latency Distribution",
    "Max Prompt Words", "Latency (s)",
    "Max Prompt Words", "RT Whisper",
)